# Install Dependencies

In [ ]:
!pip install -q --no-deps xformers trl peft accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.1/117.1 MB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 375.8/375.8 kB 35.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.0/67.0 MB 36.9 MB/s eta 0:00:00


In [ ]:
!pip install -q datasets
!pip install -q regex

In [ ]:
!pip install -q emoji
!pip install -q PyArabic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 590.6/590.6 kB 34.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.4/126.4 kB 11.1 MB/s eta 0:00:00


In [ ]:
!pip install -q diffusers

# login

In [ ]:
import huggingface_hub
huggingface_hub.login('HF_TOKEN')

# Import Required Modules

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ['CUDA_LAUNCH_BLOCKING']="1"
os.environ['TORCH_USE_CUDA_DSA'] = "1"

In [ ]:
import warnings
warnings.filterwarnings("ignore")

In [ ]:
import numpy as np
import pandas as pd
import random
from sklearn.utils import shuffle
import os
import re
from tqdm import tqdm
import bitsandbytes as bnb
import torch
import torch.nn as nn
import transformers
from datasets import Dataset
from peft import LoraConfig, PeftConfig
from trl import SFTTrainer
from transformers import (AutoModelForCausalLM,
                          AutoTokenizer,
                          BitsAndBytesConfig,
                          TrainingArguments,
                          pipeline,
                          logging)
from sklearn.metrics import (accuracy_score,
                             classification_report,
                             precision_score,
                             recall_score,
                             f1_score,
                             confusion_matrix)
from sklearn.model_selection import train_test_split
import emoji
import pyarabic.araby as araby

In [ ]:
import pandas as pd

In [ ]:
import torch
import torch.distributed as dist

# Load Model

In [ ]:
model_name = "ALLaM-AI/ALLaM-7B-Instruct-preview"

compute_dtype = getattr(torch, "float16")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=False,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    # quantization_config=bnb_config,
    device_map={"": 0},
    trust_remote_code=True,
)

model.config.use_cache = False
model.config.pretraining_tp = 1

tokenizer = AutoTokenizer.from_pretrained(model_name,
                                          trust_remote_code=True,
                                          padding_side="left",
                                          add_eos_token=True,
                                         )

# Assign pad_token if missing
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "left"

config.json:   0%|          | 0.00/684 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/4.03G [00:00<?, ?B/s]

model-00002-of-00003.safetensors:   0%|          | 0.00/4.99G [00:00<?, ?B/s]

model-00001-of-00003.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/1.23M [00:00<?, ?B/s]

In [ ]:
pipe = pipeline(task="text-generation",
                model=model,
                tokenizer=tokenizer,
                max_new_tokens=20,
                temperature=0.2
               )

Device set to use cuda:0


# Zero Shot

In [ ]:
import pandas as pd
data = pd.read_excel('Setntiment Analysis Jais Arabic Shot.xlsx')

In [ ]:
data.shape

(501, 1)

In [ ]:
pred = []

max_length = tokenizer.model_max_length
for i in tqdm(range(len(data))):
    prompt = data.iloc[i]["prompt"]
    result = pipe(prompt[:tokenizer.model_max_length], truncation=True, pad_token_id=pipe.tokenizer.eos_token_id)
    answer = result[0]['generated_text'].split("المشاعر المتوقعة:")[-1].strip()

    pred.append(answer)

100%|██████████| 501/501 [04:06<00:00,  2.04it/s]


In [ ]:
pred_zero = pd.DataFrame()
pred_zero['Predicted'] = pred
pred_zero['Predicted'].value_counts()

,count
Predicted,
,178
إيجابي,162
سلبي,66
محايدة,56
إيجابية,10
المحايدة,6
محايد,5
إيجابي\n\nسلبي\n\nمحايد\n\nالإجابة:,3
سلبية,2


In [ ]:
nor_pre = []
for pr in pred_zero['Predicted']:
  if "سلبي" in pr:
    nor_pre.append("Negative")
  elif "سلبية" in pr:
    nor_pre.append("Negative")
  elif "إيجابي" in pr:
    nor_pre.append("Positive")
  elif "إيجابية" in pr:
    nor_pre.append("Positive")
  elif "ايجابي" in pr:
    nor_pre.append("Positive")
  elif "ايجابية" in pr:
    nor_pre.append("Positive")
  elif "محايد" in pr:
    nor_pre.append("Neutral")
  elif "محايدة" in pr:
    nor_pre.append("Neutral")
  else:
    nor_pre.append("Unclassified")

In [ ]:
pred_zero['Normalized Prediction'] = nor_pre

In [ ]:
pred_zero['Normalized Prediction'].value_counts()

,count
Normalized Prediction,
Unclassified,178
Positive,176
Negative,80
Neutral,67


In [ ]:
pred_zero['prompt'] = data['prompt']

In [ ]:
pred_zero.to_csv('Sentiment Analysis Allam Zero Shot.xlsx', index = False)

In [ ]:
true = pd.read_excel('sampled_sentiment_data.xlsx')
y_true = true['sentiment'].values
print(classification_report(y_true, pred_zero['Normalized Prediction'].values, digits = 4))

              precision    recall  f1-score   support

    Negative     0.7250    0.3473    0.4696       167
     Neutral     0.7164    0.2874    0.4103       167
    Positive     0.5966    0.6287    0.6122       167
Unclassified     0.0000    0.0000    0.0000         0

    accuracy                         0.4212       501
   macro avg     0.5095    0.3159    0.3730       501
weighted avg     0.6793    0.4212    0.4974       501



# Pred Few Shot

In [ ]:
data2 = pd.read_excel('Sentiment Analysis Jais Arabic Shot.xlsx')

In [ ]:
pred = []

max_length = tokenizer.model_max_length
for i in tqdm(range(len(data2))):
    prompt = data2.iloc[i]["prompt"]
    result = pipe(prompt[:tokenizer.model_max_length], truncation=True, pad_token_id=pipe.tokenizer.eos_token_id)
    answer = result[0]['generated_text'].split("المشاعر المتوقعة:")[-1].strip()

    pred.append(answer)

100%|██████████| 501/501 [07:14<00:00,  1.15it/s]


In [ ]:
pred_few = pd.DataFrame()
pred_few['Predicted'] = pred
pred_few['Predicted'].value_counts()

,count
Predicted,
سلبي,202
إيجابي,167
محايد,128
الإجابة: سلبي,1
الإجابة: إيجابي,1
إجابتي هي: محايد,1
,1


In [ ]:
nor_pre = []
for pr in pred_few['Predicted']:
  if "سلبي" in pr:
    nor_pre.append("Negative")
  elif "سلبية" in pr:
    nor_pre.append("Negative")
  elif "إيجابي" in pr:
    nor_pre.append("Positive")
  elif "إيجابية" in pr:
    nor_pre.append("Positive")
  elif "ايجابي" in pr:
    nor_pre.append("Positive")
  elif "ايجابية" in pr:
    nor_pre.append("Positive")
  elif "محايد" in pr:
    nor_pre.append("Neutral")
  elif "محايدة" in pr:
    nor_pre.append("Neutral")
  else:
    nor_pre.append("Unclassified")

In [ ]:
pred_few['Normalized Prediction'] = nor_pre
pred_few['Normalized Prediction'].value_counts()

,count
Normalized Prediction,
Negative,203
Positive,168
Neutral,129
Unclassified,1


In [ ]:
pred_few['prompt'] = data2['prompt']

In [ ]:
pred_few.to_excel('Sentiment Analysis Allam Few Shot.xlsx', index = False)

In [ ]:
print(classification_report(y_true, pred_few['Normalized Prediction'].values, digits = 4))

              precision    recall  f1-score   support

    Negative     0.6927    0.8503    0.7634       167
     Neutral     0.6320    0.4731    0.5411       167
    Positive     0.7193    0.7365    0.7278       167

    accuracy                         0.6866       501
   macro avg     0.6813    0.6866    0.6774       501
weighted avg     0.6813    0.6866    0.6774       501



# CoT

In [ ]:
pipe = pipeline(task="text-generation",
                model=model,
                tokenizer=tokenizer,
                max_new_tokens=80,
                temperature=0.2
               )

Device set to use cuda:0


In [ ]:
data3 = pd.read_excel('Sentiment Analysis Arabic CoT.xlsx')

In [ ]:
pred = []

max_length = tokenizer.model_max_length
for i in tqdm(range(len(data3))):
    prompt = data3.iloc[i]["prompt"]
    result = pipe(prompt[:tokenizer.model_max_length], truncation=True, pad_token_id=pipe.tokenizer.eos_token_id)
    answer = result[0]['generated_text'].split("المشاعر المتوقعة:")[-1].strip()

    pred.append(answer)

100%|██████████| 501/501 [37:18<00:00,  4.47s/it]


In [ ]:
pred_cot = pd.DataFrame()
pred_cot['Predicted'] = pred
pred_cot['Predicted'].value_counts()

,count
Predicted,
سلبي,182
إيجابي,147
محايد,89
,2
محايد\n\nالجملة الذي عليك تصنيفها\nالجملة: \n#الغاء_قسم_العوايل بإذن الله خساير للمطاعم,1
...,...
محايد\n\nالجملة الذي عليك تصنيفها\nالجملة: \n@altaysercom السلام عليكم كم اسعار فنادق مكة خلال هذا الشهر,1
محايد\n\nالجملة الذي عليك تصنيفها\nالجملة: \nالسلام عليكم @stc_ksa ما هي آلية نقل خطي من @Mobily1100 إلى شركتكم؟ وماهي عروض المفوتر عندكم جزاكم الله خير,1
محايد\n\nالجملة الذي عليك تصنيفها\nالجملة: \nفاطمة محمد آدم محمد قد حفظ من سورة التحريم الآية 4 إلى سورة التحريم الآية 5 وراجع من سورة التكاثر إلى سورة القدر في شهر 6 سنة 1441,1


In [ ]:
nor_pre = []
for pr in pred_cot['Predicted']:
  if "سلبي" in pr:
    nor_pre.append("Negative")
  elif "سلبية" in pr:
    nor_pre.append("Negative")
  elif "إيجابي" in pr:
    nor_pre.append("Positive")
  elif "إيجابية" in pr:
    nor_pre.append("Positive")
  elif "ايجابي" in pr:
    nor_pre.append("Positive")
  elif "ايجابية" in pr:
    nor_pre.append("Positive")
  elif "محايد" in pr:
    nor_pre.append("Neutral")
  elif "محايدة" in pr:
    nor_pre.append("Neutral")
  else:
    nor_pre.append("Unclassified")

In [ ]:
pred_cot['Normalized Prediction'] = nor_pre
pred_cot['Normalized Prediction'].value_counts()

,count
Normalized Prediction,
Negative,198
Neutral,152
Positive,149
Unclassified,2


In [ ]:
pred_cot.to_csv('Sentiment Analysis Allam CoT.xlsx', index = False)

In [ ]:
print(classification_report(y_true, pred_cot['Normalized Prediction'].values, digits = 4))

              precision    recall  f1-score   support

    Negative     0.6212    0.7365    0.6740       167
     Neutral     0.5461    0.4970    0.5204       167
    Positive     0.7315    0.6527    0.6899       167
Unclassified     0.0000    0.0000    0.0000         0

    accuracy                         0.6287       501
   macro avg     0.4747    0.4716    0.4711       501
weighted avg     0.6329    0.6287    0.6281       501

